# 대피도 인식 모델 학습 (Colab)

층별 피난안내도 사진에서 **비상구·계단·엘리베이터·소화기·소화전·현재위치·문·실** 8가지를 찾는
탐지 모델을 학습한다. 학습이 끝나면 탐지 결과를 앱의 도면 그래프(JSON)로 바꿔,
도면 편집기(`architect.html`)의 **가져오기**로 바로 열 수 있다.

> 이 모델은 **사람이 검수할 초안**을 만든다. 대피 안내는 틀리면 사람이 다치므로,
> 모델이 만든 그래프를 그대로 쓰지 말고 편집기에서 확인한 뒤 활성화할 것.

**런타임 → 런타임 유형 변경 → GPU (T4)** 로 바꾼 뒤 위에서부터 차례로 실행하면 된다.

In [1]:
# 1. GPU 확인 — 'Tesla T4' 같은 이름이 보여야 한다. 안 보이면 런타임 유형을 GPU로 바꾼다.
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 (CPU로도 되지만 몇 배 느리다)"

Tesla T4, 15360 MiB


In [2]:
# 2. 설치 — ultralytics(YOLO)와 한글 폰트
#    폰트가 없으면 데이터 생성기가 실 이름을 영문으로 적어 실제 안내도와 멀어진다.
!pip install -q ultralytics
!apt-get -qq install -y fonts-nanum > /dev/null
!fc-cache -f > /dev/null 2>&1

import ultralytics
ultralytics.checks()

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.1/112.6 GB disk)


## 3. 코드와 데이터 올리기

아래 셋 중 편한 방법 하나만 쓰면 된다.

1. **깃 저장소가 있으면** `GIT_URL` 에 주소를 적는다.
2. **없으면** `ml/` 폴더를 통째로 zip으로 묶어 올린다 (파일 선택 창이 뜬다).
3. 데이터셋(`dataset/`)이 없어도 괜찮다 — `generate_dataset.py` 가 **같은 시드로 똑같은 100장**을
   다시 만든다. 스크립트 몇 KB만 올려도 된다.

In [3]:
import os, sys, zipfile, glob, shutil
from pathlib import Path

GIT_URL = ""   # 예: "https://github.com/gachonO2/fire.git"  — 없으면 빈 문자열로 둔다
WORK = Path("/content/work")
WORK.mkdir(exist_ok=True)

def find_ml_dir(root: Path):
    """generate_dataset.py 가 들어 있는 폴더를 찾는다."""
    hits = list(root.rglob("generate_dataset.py"))
    return hits[0].parent if hits else None

ML = find_ml_dir(WORK)

if ML is None and GIT_URL:
    !git clone --depth 1 {GIT_URL} {WORK}/repo
    ML = find_ml_dir(WORK)

if ML is None:
    from google.colab import files
    print("ml 폴더를 zip으로 묶어 올려주세요 (dataset 폴더는 없어도 됩니다).")
    for name in files.upload():
        if name.endswith(".zip"):
            with zipfile.ZipFile(name) as z:
                z.extractall(WORK)
        else:
            shutil.move(name, WORK / name)
    ML = find_ml_dir(WORK)

assert ML is not None, "generate_dataset.py 를 찾지 못했습니다. ml 폴더를 올려주세요."
os.chdir(ML)
sys.path.insert(0, str(ML))
print("작업 폴더:", ML)
print("파일:", sorted(p.name for p in ML.iterdir()))

ml 폴더를 zip으로 묶어 올려주세요 (dataset 폴더는 없어도 됩니다).


KeyboardInterrupt: 

In [ ]:
# 4. 데이터셋 준비 — 이미 있으면 그대로 쓰고, 없으면 같은 시드로 다시 만든다.
#    장수를 늘리고 싶으면 --count 만 올리면 된다 (500장까지는 몇 분이면 만들어진다).
DATASET = ML / "dataset"
COUNT = 100

if not (DATASET / "images" / "train").exists():
    !python generate_dataset.py --count {COUNT} --preview 8
else:
    print("이미 있는 데이터셋을 씁니다:", DATASET)

for split in ("train", "val", "test"):
    n = len(glob.glob(str(DATASET / "images" / split / "*.jpg")))
    print(f"  {split:>5}: {n}장")

In [ ]:
# 5. data.yaml 을 절대경로로 다시 쓴다 (상대경로는 실행 위치에 따라 어긋난다)
from symbols import NAMES

DATA_YAML = ML / "data_colab.yaml"
DATA_YAML.write_text(
    f"path: {DATASET}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES)),
    encoding="utf-8",
)
print(DATA_YAML.read_text(encoding="utf-8"))

In [ ]:
# 6. 라벨이 제대로 붙었는지 눈으로 먼저 본다.
#    학습이 안 될 때 원인의 대부분은 모델이 아니라 라벨이다.
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches
from symbols import COLOR, NAMES

samples = sorted(glob.glob(str(DATASET / "images" / "train" / "*.jpg")))[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, img_path in zip(axes.ravel(), samples):
    img = Image.open(img_path)
    ax.imshow(img); ax.set_title(Path(img_path).name); ax.axis("off")
    label_path = str(img_path).replace("/images/", "/labels/").replace(".jpg", ".txt")
    W, H = img.size
    for line in open(label_path, encoding="utf-8"):
        c, cx, cy, w, h = line.split()
        c, cx, cy, w, h = int(c), float(cx), float(cy), float(w), float(h)
        col = tuple(v / 255 for v in COLOR[NAMES[c]])
        ax.add_patch(patches.Rectangle(((cx - w / 2) * W, (cy - h / 2) * H),
                                       w * W, h * H, lw=1.6, edgecolor=col, facecolor="none"))
plt.tight_layout(); plt.show()

## 7. 학습

좌우 뒤집기(`fliplr`)는 꺼 둔다. 도면을 좌우로 뒤집으면 글자와 비상구 화살표가 거울상이 되어
현실에 없는 그림이 되고, 모델이 방향을 잘못 배운다. 회전·크기·색 변형은 데이터 생성기가
이미 넣어 두었고, 여기서 조금 더 준다.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")   # 안 받아지면 "yolov8n.pt" 로 바꾼다

results = model.train(
    data=str(DATA_YAML),
    epochs=150,
    imgsz=800,        # 소화기 같은 작은 기호가 20px 남짓이라 640보다 크게 본다
    batch=8,          # T4 기준. 메모리 부족(CUDA out of memory)이면 4로 줄인다
    patience=40,      # 40에폭 동안 좋아지지 않으면 멈춘다
    degrees=5.0,
    translate=0.08,
    scale=0.35,
    fliplr=0.0,       # 도면을 좌우로 뒤집으면 글자가 거울상이 된다
    flipud=0.0,
    mosaic=0.7,
    close_mosaic=20,
    project="runs",
    name="evac",
    seed=0,
)
print("가중치:", results.save_dir)

In [ ]:
# 8. 검증 — 클래스별로 어디가 약한지 본다.
#    mAP50 이 아니라 **클래스별 재현율(recall)** 을 먼저 봐야 한다.
#    비상구를 하나 놓치면 그 층의 대피 안내가 통째로 틀어지기 때문이다.
best = Path(results.save_dir) / "weights" / "best.pt"
model = YOLO(str(best))
metrics = model.val(data=str(DATA_YAML), imgsz=800, split="val")

from symbols import KOREAN, NAMES
print(f"\n전체  mAP50 {metrics.box.map50:.3f}   mAP50-95 {metrics.box.map:.3f}\n")
print(f"{'클래스':<12}{'정밀도':>8}{'재현율':>8}{'mAP50':>8}")
for i, c in enumerate(metrics.box.ap_class_index):
    name = NAMES[int(c)]
    p, r, ap50 = metrics.box.p[i], metrics.box.r[i], metrics.box.ap50[i]
    print(f"{KOREAN[name]:<12}{p:>8.3f}{r:>8.3f}{ap50:>8.3f}")

In [ ]:
# 9. 한 번도 학습에 쓰지 않은 test 이미지로 예측해 본다
test_images = sorted(glob.glob(str(DATASET / "images" / "test" / "*.jpg")))
preds = model.predict(test_images[:4], imgsz=800, conf=0.35, verbose=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
for ax, pred in zip(axes.ravel(), preds):
    ax.imshow(pred.plot()[:, :, ::-1])   # BGR → RGB
    ax.set_title(Path(pred.path).name); ax.axis("off")
plt.tight_layout(); plt.show()

## 10. 탐지 결과 → 대피 경로 그래프

여기가 이 모델을 앱에 붙이는 지점이다. 기호 위치만으로는 안내를 못 하고,
**어디서 어디로 갈 수 있는지**가 있어야 길찾기가 돈다.
`predict_to_plan.py` 는 실 사각형 바깥을 복도로 보고, 복도만 밟아 지점들을 잇는다.

In [ ]:
target = test_images[0]
!python predict_to_plan.py --model {best} --image {target} --meters-wide 40 --out /content/plan.json

import json
plan = json.load(open("/content/plan.json", encoding="utf-8"))
print(f"\n지점 {len(plan['nodes'])}개 · 통로 {len(plan['edges'])}개")
for t in ("exit", "stair", "elevator", "room", "junction"):
    print(f"  {t:<10} {sum(1 for n in plan['nodes'] if n['type'] == t)}개")

In [ ]:
# 만들어진 그래프를 도면 위에 얹어 확인한다 — 통로가 벽을 뚫고 지나가면 안 된다
from PIL import ImageDraw

img = Image.open(target).convert("RGB")
d = ImageDraw.Draw(img)
pos = {n["id"]: (n["x"], n["y"]) for n in plan["nodes"]}
for e in plan["edges"]:
    d.line([pos[e["a"]], pos[e["b"]]], fill=(255, 0, 200), width=3)
tint = {"exit": (0, 180, 60), "stair": (0, 90, 255), "elevator": (150, 0, 220),
        "room": (255, 140, 0), "junction": (60, 60, 60)}
for n in plan["nodes"]:
    x, y = pos[n["id"]]
    r = 6 if n["type"] != "junction" else 3
    d.ellipse([x - r, y - r, x + r, y + r], fill=tint[n["type"]], outline=(255, 255, 255))

plt.figure(figsize=(11, 11)); plt.imshow(img); plt.axis("off"); plt.show()

In [ ]:
# 11. 내보내기 — ONNX 는 어디서나 열리고, 브라우저에서 바로 돌리려면 tfjs 가 필요하다
model.export(format="onnx", imgsz=800, opset=12, simplify=True)

try:
    model.export(format="tfjs", imgsz=800)   # 프론트에서 직접 추론하고 싶을 때
except Exception as err:
    print("tfjs 변환 실패(의존성 문제가 잦다). ONNX 만 써도 됩니다:", err)

In [ ]:
# 12. 결과 내려받기 — best.pt(재학습용) + onnx(배포용) + 학습 그래프
import shutil
from google.colab import files

out_dir = Path("/content/evac_model")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir()

weights = Path(results.save_dir) / "weights"
for pattern in ("best.pt", "best.onnx"):
    for f in weights.glob(pattern):
        shutil.copy(f, out_dir)
for name in ("results.png", "confusion_matrix_normalized.png", "results.csv"):
    src = Path(results.save_dir) / name
    if src.exists():
        shutil.copy(src, out_dir)
shutil.copy(DATA_YAML, out_dir)

shutil.make_archive("/content/evac_model", "zip", out_dir)
print("담긴 파일:", sorted(p.name for p in out_dir.iterdir()))
files.download("/content/evac_model.zip")

## 다음 단계 — 진짜 대피도로 마무리 학습

여기까지의 모델은 **합성 도면 100장**으로 배운 것이다. 배치와 기호 규칙은 실제와 같지만,
종이 질감·유리 반사·비스듬한 촬영 각도까지는 흉내내지 못한다. 실제 건물에서 쓰려면
가천대 3층처럼 **실제로 안내할 건물의 피난안내도를 20~30장** 찍어 이어서 학습시키는 것이 가장 크게 는다.

1. 폰으로 안내도를 각도·거리·조명을 바꿔 20~30장 찍는다.
2. [Roboflow](https://roboflow.com) 나 [labelImg](https://github.com/HumanSignal/labelImg) 로
   YOLO 형식 라벨을 만든다. 클래스 순서는 `symbols.py` 와 **반드시 같게** 한다.
3. `dataset/images/train`, `dataset/labels/train` 에 합쳐 넣는다.
4. 처음부터가 아니라 방금 만든 가중치에서 이어 학습한다:

```python
model = YOLO(str(best))
model.train(data=str(DATA_YAML), epochs=60, imgsz=800, lr0=0.002, fliplr=0.0)
```

합성 데이터로 크게 잡아 두고 실제 사진으로 다듬는 순서라, 실제 사진이 몇 십 장만 있어도 된다.